# Step 7 — SageMaker Model Registry

Version and approve the trained model.

In [ ]:
import boto3
import sagemaker
from sagemaker.huggingface import HuggingFaceModel

BUCKET = "YOUR-S3-BUCKET-NAME"
REGION = "us-east-2"

boto_session = boto3.Session(region_name=REGION)
session = sagemaker.Session(boto_session=boto_session, default_bucket=BUCKET)
role = sagemaker.get_execution_role()
sm_client = boto3.client("sagemaker", region_name=REGION)

# Get model URI from the pipeline training job
training_jobs = sm_client.list_training_jobs(
    NameContains="TrainBioBERT",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1
)
job_name   = training_jobs["TrainingJobSummaries"][0]["TrainingJobName"]
job_detail = sm_client.describe_training_job(TrainingJobName=job_name)
model_uri  = job_detail["ModelArtifacts"]["S3ModelArtifacts"]
print(f"Model URI: {model_uri}")

In [ ]:
hf_model = HuggingFaceModel(
    model_data=model_uri,
    role=role,
    transformers_version="4.26",
    pytorch_version="1.13",
    py_version="py39",
    sagemaker_session=session,
)

model_package = hf_model.register(
    content_types=["application/json"],
    response_types=["application/json"],
    model_package_group_name="SymptomClassifier",
    approval_status="Approved",
    description="Bio_ClinicalBERT fine-tuned on symptom2disease (76.4% test accuracy)",
    inference_instances=["ml.m5.xlarge"],   # Required!
    transform_instances=["ml.m5.xlarge"],   # Required!
)
print(f"Model registered ✅")
print(f"ARN: {model_package.model_package_arn}")